In [1]:
import pandas as pd
import numpy as np

TARGETS = ["x"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_x,MSE_ZZx1_x,R2_ZZx2_x,MSE_ZZx2_x,R2_ZZxReto_x,MSE_ZZxReto_x
0,model_arch80_r0.01_Ld0.5_Lp0.5_seed4824,[80],0.5,0.5,0.01,4824,0.850297,0.961782,0.735775,0.856730,0.873064,0.935275
1,model_arch80_r0.01_Ld0.5_Lp0.5_seed8256,[80],0.5,0.5,0.01,8256,0.991744,0.976034,-3.085943,0.792423,0.391849,0.941736
2,model_arch80_r0.01_Ld0.5_Lp0.5_seed6040,[80],0.5,0.5,0.01,6040,0.878851,0.961907,0.866954,0.854816,0.880287,0.936522
3,model_arch80_r0.01_Ld0.5_Lp0.5_seed2393,[80],0.5,0.5,0.01,2393,0.834537,0.962117,0.866664,0.826311,0.921089,0.940711
4,model_arch80_r0.01_Ld0.5_Lp0.5_seed4073,[80],0.5,0.5,0.01,4073,0.888544,0.961025,0.872799,0.830636,0.882566,0.939119
...,...,...,...,...,...,...,...,...,...,...,...,...
626,model_arch100_r0.9_Ld0.7_Lp0.3_seed8256,[100],0.7,0.3,0.90,8256,0.836374,0.959898,0.596291,0.856686,0.877303,0.932653
627,model_arch100_r0.9_Ld0.7_Lp0.3_seed6040,[100],0.7,0.3,0.90,6040,0.891582,0.960314,0.652739,0.869204,0.824298,0.931233
628,model_arch100_r0.9_Ld0.7_Lp0.3_seed2393,[100],0.7,0.3,0.90,2393,0.868294,0.956663,0.365436,0.860571,0.828354,0.926897
629,model_arch100_r0.9_Ld0.7_Lp0.3_seed4073,[100],0.7,0.3,0.90,4073,0.904306,0.953534,0.194337,0.864086,0.806138,0.925645


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - x


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
380,model_arch92_r0.9_Ld0.3_Lp0.7_seed9339,[92],0.871519,0.909306,0.898984,0.882384
64,model_arch82_r0.01_Ld0.5_Lp0.5_seed4073,[82],0.879069,0.901868,0.887688,0.879495
65,model_arch82_r0.01_Ld0.5_Lp0.5_seed9339,[82],0.867361,0.898230,0.903428,0.878827
145,model_arch84_r0.01_Ld0.7_Lp0.3_seed9339,[84],0.887017,0.902015,0.870851,0.876203
298,model_arch89_r0.9_Ld0.7_Lp0.3_seed2393,[89],0.892922,0.917015,0.851282,0.874877



📊 MÉTRICAS COMPLETAS - TOP 5 (x)


,model,Neurons,R2_ZZx1_x,R2_ZZx2_x,R2_ZZxReto_x,R2_train_mean,R2_val_mean,R2_test_mean,Score
380,model_arch92_r0.9_Ld0.3_Lp0.7_seed9339,[92],0.871519,0.909306,0.898984,0.871519,0.909306,0.898984,0.882384
64,model_arch82_r0.01_Ld0.5_Lp0.5_seed4073,[82],0.879069,0.901868,0.887688,0.879069,0.901868,0.887688,0.879495
65,model_arch82_r0.01_Ld0.5_Lp0.5_seed9339,[82],0.867361,0.898230,0.903428,0.867361,0.898230,0.903428,0.878827
145,model_arch84_r0.01_Ld0.7_Lp0.3_seed9339,[84],0.887017,0.902015,0.870851,0.887017,0.902015,0.870851,0.876203
298,model_arch89_r0.9_Ld0.7_Lp0.3_seed2393,[89],0.892922,0.917015,0.851282,0.892922,0.917015,0.851282,0.874877


In [5]:
final_table.to_excel("BestModels-otm.xlsx")

In [6]:
# ============================================
# MÉDIA, DESVIO, MÍNIMO E MÁXIMO
# ============================================

summary_tables = {}

for target in TARGETS:

    top_df = results.copy()
    rows = []

    for s in SETS_CATEGORY.keys():

        r2_col = f"R2_{s.replace('-', '_')}_{target}"
        mse_col = f"MSE_{s.replace('-', '_')}_{target}"

        row = {
            "Set": s,
            "Category": SETS_CATEGORY[s]
        }

        # =========================
        # R²
        # =========================
        if r2_col in top_df.columns:
            row["R2_mean"] = top_df[r2_col].mean()
            row["R2_std"]  = top_df[r2_col].std()
            row["R2_min"]  = top_df[r2_col].min()
            row["R2_max"]  = top_df[r2_col].max()
        else:
            row["R2_mean"] = np.nan
            row["R2_std"]  = np.nan
            row["R2_min"]  = np.nan
            row["R2_max"]  = np.nan

        # =========================
        # MSE
        # =========================
        if mse_col in top_df.columns:
            row["MSE_mean"] = top_df[mse_col].mean()
            row["MSE_std"]  = top_df[mse_col].std()
            row["MSE_min"]  = top_df[mse_col].min()
            row["MSE_max"]  = top_df[mse_col].max()
        else:
            row["MSE_mean"] = np.nan
            row["MSE_std"]  = np.nan
            row["MSE_min"]  = np.nan
            row["MSE_max"]  = np.nan

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    summary_tables[target] = summary_df

    # =========================
    # MOSTRA SOMENTE A TABELA
    # =========================
    display(
        summary_df.style.format({
            "R2_mean": "{:.4f}",
            "R2_std":  "{:.4f}",
            "R2_min":  "{:.4f}",
            "R2_max":  "{:.4f}",

            "MSE_mean": "{:.6f}",
            "MSE_std":  "{:.6f}",
            "MSE_min":  "{:.6f}",
            "MSE_max":  "{:.6f}"
        })
    )

AttributeError: The '.style' accessor requires jinja2

In [9]:
!pip install jinja2


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
